# Project 1 — Train Word Embeddings from Scratch
TensorFlow Skip-Gram with negative sampling, training loss, 2D word map, similarity, and vector arithmetic. Run top to bottom in Google Colab (CPU).

In [ ]:
import numpy as np, tensorflow as tf, matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import random
random.seed(42); np.random.seed(42); tf.random.set_seed(42)

## 1. Training data: words learn from their neighbors
This tiny synthetic corpus is intentionally repetitive. Real word embeddings require much more diverse text.

In [ ]:
groups={'animals':['cat','dog','kitten','puppy'],'fruits':['apple','banana','orange','grape'],'vehicles':['car','truck','bus','train']}
contexts={'animals':['pet','fur','play','sleep'],'fruits':['sweet','fresh','eat','ripe'],'vehicles':['drive','road','wheel','travel']}
sentences=[]
for category,words in groups.items():
    for _ in range(60):
        for word in words: sentences.append([word]+random.sample(contexts[category],2))
random.shuffle(sentences)
vocab=sorted(set(w for s in sentences for w in s)); ids={w:i for i,w in enumerate(vocab)}
print('Vocabulary:',vocab);print('Example sentences:',sentences[:5])

## 2. Positive and negative Skip-Gram pairs
Positive pairs occur together; negative pairs are randomly sampled non-occurring pairs.

In [ ]:
positive=[(ids[s[0]],ids[w],1) for s in sentences for w in s[1:]]
known={(a,b) for a,b,_ in positive}; rng=np.random.default_rng(42); examples=list(positive)
for center,context,_ in positive:
    for _ in range(2):
        neg=int(rng.integers(len(vocab)))
        while (center,neg) in known: neg=int(rng.integers(len(vocab)))
        examples.append((center,neg,0))
rng.shuffle(examples)
x=np.array([[a,b] for a,b,_ in examples],dtype='int32'); y=np.array([label for _,_,label in examples],dtype='float32')
print('Training examples:',len(y))

## 3. Build and train a two-dimensional model
Separate center/context embedding matrices learn which pairs are plausible.

In [ ]:
dim=2
center=tf.keras.Input(shape=(),dtype='int32'); context=tf.keras.Input(shape=(),dtype='int32')
center_table=tf.keras.layers.Embedding(len(vocab),dim,name='word_embeddings')
context_table=tf.keras.layers.Embedding(len(vocab),dim,name='context_embeddings')
a=center_table(center);b=context_table(context)
score=tf.keras.layers.Dot(axes=-1)([a,b]); output=tf.keras.layers.Activation('sigmoid')(score)
model=tf.keras.Model([center,context],output)
model.compile(optimizer=tf.keras.optimizers.Adam(0.03),loss='binary_crossentropy')
initial=center_table.get_weights()[0].copy()
def plot_map(vectors,title):
    plt.figure(figsize=(9,6))
    for group,words in groups.items():
        for w in words:
            u,v=vectors[ids[w]];plt.scatter(u,v,label=group if w==words[0] else None);plt.annotate(w,(u,v),xytext=(5,5),textcoords='offset points')
    plt.axhline(0,color='gray',lw=.5);plt.axvline(0,color='gray',lw=.5)
    plt.title(title);plt.legend();plt.grid(alpha=.2);plt.show()
plot_map(initial,'Before training: random embeddings')
history=model.fit([x[:,0],x[:,1]],y,epochs=35,batch_size=128,verbose=0)
vectors=center_table.get_weights()[0]
plt.plot(history.history['loss']);plt.title('Training loss');plt.xlabel('Epoch');plt.show()
plot_map(vectors,'After training: learned embeddings')

## 4. Nearest words and cosine similarity
Try changing the corpus and rerunning to see the map move.

In [ ]:
def nearest(vec,k=5,exclude=()):
    sims=cosine_similarity(np.asarray(vec).reshape(1,-1),vectors)[0]
    return [(vocab[i],round(float(sims[i]),3)) for i in np.argsort(-sims) if vocab[i] not in exclude][:k]
for w in ['cat','apple','car']:print(w,nearest(vectors[ids[w]],exclude={w}))

## 5. Vector arithmetic (Experiment 3)
Word analogies are NOT guaranteed in a tiny 2D model. Treat them as hypotheses to test.

In [ ]:
def analogy(a,b,c):
    target=vectors[ids[a]]-vectors[ids[b]]+vectors[ids[c]]
    return nearest(target,exclude={a,b,c})
print('puppy - dog + cat:',analogy('puppy','dog','cat'))
print('banana - apple + orange:',analogy('banana','apple','orange'))

## Student challenges
1. Increase training epochs: what changes?
2. Change `dim=2` to `dim=8` (skip 2D plotting): do nearest neighbors improve?
3. Give `apple` animal-like contexts. Does it move toward animals?
4. Why does low training loss not guarantee meaningful analogies?

**Teacher note:** uniform negative sampling is a simplified teaching version of Word2Vec.